# Battle RL — 通用远程 PPO GPU Worker

连接到本地 hub 的 cloudflared tunnel，以 **pull** 或 **push** 模式领取任意 PPO 任务。

## 两种模式

| 模式 | 说明 | 适用场景 |
|---|---|---|
| **pull**（默认） | Worker 轮询 hub 领取 job（hub 侧 cloudflared） | 标准的 Kaggle/Colab 接入方式 |
| **push** | Worker 启动服务端 + 本地 cloudflared，hub 主动推送 job | hub 侧不方便跑 tunnel 时 |

## 使用前

1. 确保 hub 端已启动：`bun tools/hub-start.ts <course>`（pull 模式）或 hub 已配置 push 节点
2. 在下方的 `⚙️ 参数配置` 单元格填入连接信息
3. 依次运行各单元格

> **通用性**：本 notebook 不绑定任何特定课程——hub 发布的 job manifest 携带完整课程上下文，
> worker 按 manifest 中的 reward 公式和超参执行 PPO。

---
## ⚙️ 参数配置

In [ ]:
# @title 填入连接参数
import os
import sys
import time

# ── 模式选择 ──
MODE = "pull"            # "pull" 或 "push"

# ── Pull 模式参数（MODE="pull"） ──
# HUB_URL 和 HUB_TOKEN 由 hub 端提供：
#   - 运行 hub-start.ts 后，终端会打印 "📋 Kaggle 粘贴用" 的 URL 和 token
#   - 或从 rl-config.json 的 rl.remote_hub_url / rl.remote_token 获取
HUB_URL = "https://your-tunnel.trycloudflare.com"
HUB_TOKEN = "YOUR_TOKEN_HERE"

# ── Push 模式参数（MODE="push"） ──
# 本机运行 cloudflared 暴露端口，hub 侧 push job 到此节点
PUSH_PORT = 8790
PUSH_TOKEN = "YOUR_TOKEN_HERE"
# cloudflared 路径（通常自动在 PATH 中，可留空）
CLOUDFLARED_PATH = ""

# ── 通用参数 ──
# 保活时长：Kaggle GPU 最长 9h，Colab 免费版约 90min 空闲回收
MAX_SESSION_HOURS = 9
POLL_INTERVAL_SEC = 5

PLATFORM = "colab" if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ else "kaggle"

print(f"[{time.strftime('%H:%M:%S')}] Mode = {MODE}")
if MODE == "pull":
    print(f"[{time.strftime('%H:%M:%S')}] HUB_URL = '{HUB_URL}'")
    print(f"[{time.strftime('%H:%M:%S')}] HUB_TOKEN len = {len(HUB_TOKEN)}")
else:
    print(f"[{time.strftime('%H:%M:%S')}] PUSH_PORT = {PUSH_PORT}")
    print(f"[{time.strftime('%H:%M:%S')}] PUSH_TOKEN len = {len(PUSH_TOKEN)}")
print(f"[{time.strftime('%H:%M:%S')}] Platform = {PLATFORM}")
print(f"[{time.strftime('%H:%M:%S')}] Max session = {MAX_SESSION_HOURS}h")

---
## 1. 安装依赖

Kaggle/Colab 默认环境已预装 PyTorch（CUDA 版），仅需确认版本并按需补齐。

In [ ]:
import subprocess

import torch


def run(cmd, **kw):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kw)


# 确认 torch 可用
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[{time.strftime('%H:%M:%S')}] torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[{time.strftime('%H:%M:%S')}]   device: {torch.cuda.get_device_name(0)}")
    print(f"[{time.strftime('%H:%M:%S')}]   CUDA capability: {torch.cuda.get_device_capability()}")

# 若 torch 版本过旧，可升级
if int(torch.__version__.split('.')[0]) < 2:
    print(f"[{time.strftime('%H:%M:%S')}] torch 版本过旧，升级中...")
    run(f"{sys.executable} -m pip install --quiet --upgrade torch")

print(f"[{time.strftime('%H:%M:%S')}] Dependencies ready")

---
## 2. 会话保活

Kaggle GPU 会话最长 9h，每周配额 30h；Colab 免费版约 90min 空闲回收。
通过定期输出（Kaggle）或模拟点击（Colab）防止超时回收。

In [ ]:
import threading

KEEPALIVE_STOP = threading.Event()

if PLATFORM == "colab":
    from IPython.display import Javascript
    from IPython.display import display as ipy_display

    def keepalive_loop():
        """每 60s 点一次 Colab 的 connect 按钮防止超时。"""
        n = 0
        while not KEEPALIVE_STOP.is_set():
            try:
                ipy_display(Javascript("""
                    function clickConnect() {
                        document.querySelector("colab-connect-button")?.click();
                    }
                    setTimeout(clickConnect, 1000);
                """))
                n += 1
            except Exception:
                pass
            KEEPALIVE_STOP.wait(60)
        print(f"[{time.strftime('%H:%M:%S')}] [keepalive] stopped after {n} pings")
else:
    def keepalive_loop():
        """每 120s 打印一次心跳，防止 Kaggle 空闲回收。"""
        n = 0
        while not KEEPALIVE_STOP.is_set():
            print(f"[{time.strftime('%H:%M:%S')}] [keepalive] alive ({n * 2} min elapsed)")
            n += 1
            KEEPALIVE_STOP.wait(120)
        print(f"[{time.strftime('%H:%M:%S')}] [keepalive] stopped after {n} pings ({n * 2} min)")

th = threading.Thread(target=keepalive_loop, daemon=True, name="keepalive")
th.start()
print(f"[{time.strftime('%H:%M:%S')}] Keepalive thread started (every {'60' if PLATFORM == 'colab' else '120'}s, max {MAX_SESSION_HOURS}h)")

---
## 3. 定义 Worker 方法

定义 pull 和 push 两种模式的 GPU worker 方法，供下一步启动时调用。

**Pull 模式**：从 hub 下载 code.zip → 解压到 `sys.path` → 启动 `worker_loop` 轮询领取 job。

**Push 模式**：启动 `remote_worker_serve` HTTP 服务 + cloudflared 隧道暴露到公网，
hub 侧主动推送 job 到此节点。

In [ ]:
def _log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] {msg}", flush=True)


# ────────────────────────────────────────────── Pull 模式 ──

def run_pull_worker() -> int:
    """以 pull 模式连接 hub 轮询 PPO job，返回处理 job 数。"""
    import io
    import urllib.error
    import urllib.request
    import zipfile
    from pathlib import Path

    work_dir = Path("/tmp/remote-worker")
    work_dir.mkdir(parents=True, exist_ok=True)

    _log(f"Downloading code.zip from {HUB_URL}/code...")
    req = urllib.request.Request(
        f"{HUB_URL.rstrip('/')}/code",
        headers={"Authorization": f"Bearer {HUB_TOKEN}"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            code_raw = resp.read()
            _log(f"code.zip: {len(code_raw)} bytes (HTTP {resp.status})")
    except urllib.error.HTTPError as e:
        _log(f"FAILED: HTTP {e.code} — {e.read().decode()[:200]}")
        raise SystemExit(1) from None
    except Exception as e:
        _log(f"FAILED: {e}")
        raise SystemExit(1) from None

    code_dir = Path("/tmp/worker-code")
    code_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(code_raw)) as z:
        z.extractall(code_dir)
    sys.path.insert(0, str(code_dir))
    _log(f"code.zip extracted -> {code_dir} (sys.path[0])")

    from remote.worker import worker_loop

    _log(f"Testing connectivity to hub...")
    req = urllib.request.Request(
        f"{HUB_URL.rstrip('/')}/ping",
        headers={"Authorization": f"Bearer {HUB_TOKEN}"},
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            _log(f"hub HTTP {resp.status}")
    except Exception as e:
        _log(f"CONNECTION FAILED: {e}")
        raise SystemExit(1) from None

    max_idle = max(3600, (MAX_SESSION_HOURS - 1) * 3600)
    _log(f"Starting worker loop (max_idle={max_idle}s, poll_sec={POLL_INTERVAL_SEC}s)...")

    try:
        n = worker_loop(
            HUB_URL,
            HUB_TOKEN,
            work_dir=work_dir,
            device=DEVICE,
            torch_threads=0,
            poll_sec=POLL_INTERVAL_SEC,
            once=False,
            max_idle_sec=max_idle,
        )
        return n
    except KeyboardInterrupt:
        _log(f"Worker interrupted by user")
        return 0


# ────────────────────────────────────────────── Push 模式 ──

def run_push_worker() -> int:
    """以 push 模式启动 worker_server + cloudflared 隧道，等待 hub 推送 job。"""
    import re
    import subprocess as _sp
    from pathlib import Path

    work_dir = Path("/tmp/remote-worker-serve")
    work_dir.mkdir(parents=True, exist_ok=True)

    # ── 查找 / 自动安装 cloudflared ──
    cf_bin = CLOUDFLARED_PATH or _sp.getoutput("where cloudflared 2>nul || which cloudflared 2>/dev/null").strip()
    if not cf_bin:
        _log(f"cloudflared 未找到，尝试自动安装...")
        try:
            cf_install_dir = Path("/usr/local/bin")
            cf_install_dir.mkdir(parents=True, exist_ok=True)
            cf_bin = str(cf_install_dir / "cloudflared")
            _sp.run(
                ["curl", "-fsSL",
                 "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                 "-o", cf_bin],
                check=True, timeout=60,
            )
            os.chmod(cf_bin, 0o755)
            _log(f"cloudflared 已安装到 {cf_bin}")
        except Exception as e:
            _log(f"cloudflared 自动安装失败: {e}")
            cf_bin = ""

    if not cf_bin:
        _log(f"WARNING: cloudflared 不可用——push 模式需要隧道暴露服务")
        _log(f"请手动安装后重试：")
        _log(f"  # Linux: curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
        _log(f"  # macOS: brew install cloudflared")
        _log(f"  # 然后启动: cloudflared tunnel --url http://localhost:{PUSH_PORT}")

    # ── 启动 worker_server ──
    _log(f"Starting worker server on 0.0.0.0:{PUSH_PORT}...")
    serve_log = work_dir / "serve.log"
    with open(serve_log, "w") as log_f:
        serve_proc = _sp.Popen(
            [sys.executable, "-u", "-m", "remote_worker_serve",
             "--port", str(PUSH_PORT),
             "--token", PUSH_TOKEN,
             "--work", str(work_dir),
             "--device", DEVICE],
            stdout=log_f, stderr=_sp.STDOUT,
        )
    _log(f"Worker server started (PID {serve_proc.pid})")

    # 等待服务就绪
    import urllib.request as _ur

    def _ping_ok() -> bool:
        try:
            req = _ur.Request(
                f"http://127.0.0.1:{PUSH_PORT}/ping",
                headers={"Authorization": f"Bearer {PUSH_TOKEN}"},
            )
            with _ur.urlopen(req, timeout=5) as r:
                return r.status == 200
        except Exception:
            return False

    t0 = time.time()
    while time.time() - t0 < 30:
        if _ping_ok():
            break
        time.sleep(1)
    if _ping_ok():
        _log(f"Worker server ready")
    else:
        _log(f"Worker server NOT ready (30s timeout) — check {serve_log}")
        serve_proc.kill()
        return -1

    # ── 启动 cloudflared tunnel ──
    cf_url = None
    cf_proc = None
    if cf_bin:
        _log(f"Starting cloudflared tunnel...")
        cf_log = work_dir / "cloudflared.log"
        with open(cf_log, "w") as log_f:
            cf_proc = _sp.Popen(
                [cf_bin, "tunnel", "--url", f"http://localhost:{PUSH_PORT}", "--logfile", str(cf_log)],
                stdout=log_f, stderr=_sp.STDOUT,
            )

        t0 = time.time()
        while time.time() - t0 < 60:
            try:
                text = cf_log.read_text(encoding="utf-8", errors="replace")
                urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
                if urls:
                    cf_url = urls[-1]
                    break
            except Exception:
                pass
            if cf_proc.poll() is not None:
                _log(f"cloudflared exited early (code {cf_proc.returncode})")
                break
            time.sleep(2)

        if cf_url:
            _log(f"cloudflared tunnel URL: {cf_url}")
            _log(f"将此 URL 配置到 hub 侧 rl-config.json 的 nodes 条目")
        else:
            _log(f"cloudflared tunnel URL not obtained (timeout/error)")
    else:
        _log(f"cloudflared 不可用，跳过隧道启动")

    # ── 等待 job（保持会话活跃） ──
    _log(f"Waiting for jobs... (keepalive active)")
    _log(f"hub 侧配置 push 节点后，job 会自动推送至此")

    t_start = time.time()
    try:
        while True:
            if serve_proc.poll() is not None:
                _log(f"Worker server exited (code {serve_proc.returncode})")
                break
            if time.time() - t_start > MAX_SESSION_HOURS * 3600:
                _log(f"Max session reached ({MAX_SESSION_HOURS}h)")
                break
            time.sleep(30)
    except KeyboardInterrupt:
        _log(f"Interrupted by user")
    finally:
        if cf_proc and cf_proc.poll() is None:
            cf_proc.kill()
            _log(f"cloudflared stopped")
        if serve_proc.poll() is None:
            serve_proc.kill()
            _log(f"Worker server stopped")

    return 0

---
## 4. 启动 GPU Worker

根据 `MODE` 选择启动 pull 或 push 模式的 GPU worker。
可以随时中断此单元格（`Kernel → Interrupt` 或 `Runtime → Interrupt execution`），worker 会优雅退出。
中断后如还有配额，可重新运行此单元格继续工作。

In [ ]:
t_start = time.time()

if MODE == "pull":
    print(f"\n{'='*60}")
    print(f"  [battle-rl] 工作模式: PULL")
    print(f"  [battle-rl] 连接 hub: {HUB_URL}")
    print(f"  [battle-rl] 平台: {PLATFORM}")
    print(f"  [battle-rl] 设备: {DEVICE}")
    print(f"  [battle-rl] 最大会话: {MAX_SESSION_HOURS}h")
    print(f"  [battle-rl] 行为: 轮询 hub 领取 PPO job → 执行 → 回传结果")
    print(f"{'='*60}\n")

    n = run_pull_worker()

    elapsed = time.time() - t_start
    print(f"\n[{time.strftime('%H:%M:%S')}] [battle-rl] Worker exited: {n} job(s) processed")
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] Session duration: {elapsed/60:.1f} min")

elif MODE == "push":
    print(f"\n{'='*60}")
    print(f"  [battle-rl] 工作模式: PUSH")
    print(f"  [battle-rl] 监听端口: {PUSH_PORT}")
    print(f"  [battle-rl] 平台: {PLATFORM}")
    print(f"  [battle-rl] 设备: {DEVICE}")
    print(f"  [battle-rl] 最大会话: {MAX_SESSION_HOURS}h")
    print(f"  [battle-rl] 行为: 启动 worker_server + cloudflared → 等待 hub 推送 job")
    print(f"{'='*60}\n")

    ret = run_push_worker()

    elapsed = time.time() - t_start
    print(f"\n[{time.strftime('%H:%M:%S')}] [battle-rl] Session duration: {elapsed/60:.1f} min")

else:
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] 未知 MODE={MODE!r}，可选 'pull' 或 'push'")

# 停止保活
KEEPALIVE_STOP.set()

---
## 5. 停止保活（备用）

如果提前中断了步骤 4，运行此单元格停止保活线程。
正常退出时保活会自动停止，无需手动运行此单元格。

In [ ]:
KEEPALIVE_STOP.set()
print(f"[{time.strftime('%H:%M:%S')}] Keepalive stopped")

---
## 6. 会话管理与故障排查

### GPU 配额管理

| 平台 | 限制 | 说明 |
|---|---|---|
| Kaggle | 每周 30h GPU，单次 9h | 每周一 UTC 重置，worker 空闲超时自动退出 |
| Colab 免费 | ~90min 空闲回收 | 保活线程每 60s 模拟点击 |
| Colab Pro | 更长的使用时间 | 同上，保活线程同样适用 |

### 中断后重连

- 中断后重新运行步骤 4 即可继续
- hub 的 job 幂等机制保证不会重复训练同一轮

### 预期日志

**hub 端**（pull 模式）：
```
[hub-server] "GET /jobs/next HTTP/1.1" 200 -
[hub-server] "GET /jobs/{id}/payload HTTP/1.1" 200 -
[hub-server] "POST /jobs/{id}/result HTTP/1.1" 200 -
```

**worker 端**（pull 模式）：
```
[worker] job {id} claimed — downloading payload
[worker] job {id}: code.zip unpacked (N bytes, M .py files) -> sys.path[0]
[worker] job {id}: PPO done in {sec}s, steps={n} chunks={m} kl={k}
[worker] job {id} done — result accepted
```

**hub 端**（push 模式）：
```
[push] job {id} 已推送到 {node_url}
[push] wait_result: job {id} done
[hub] weights landed -> {out_weights}
```

**worker 端**（push 模式）：
```
[worker-serve] job {id} accepted
[worker-serve] job {id}: PPO done in {sec}s
[worker-serve] job {id} done — result ready for pickup
```

### 常见问题

| 症状 | 原因 | 处理 |
|---|---|---|
| `CONNECTION FAILED` | hub tunnel 未启动或 URL 过期 | 确认 hub 端 tunnel 运行中，更新 HUB_URL |
| `HTTP 401` | token 不匹配 | 检查凭证与 hub 端 rl-config.json 一致 |
| `payload_sha256 不匹配` | 传输损坏 | 自动重试，hub 的 job 幂等机制保证不重复 |
| 长时间无 job | rollout 采集未完成 | hub 端 rollout 采集完成后自动发布 job |
| Kaggle 配额耗尽 | 30h/周 GPU 用完 | 等待周一重置，或使用 Colab Pro/AutoDL 替代 |
| push 模式 cloudflared 启动失败 | 镜像缺少 cloudflared | 手动安装，或改用 pull 模式 |

---
## 附录：与 hub-start.ts 的配合

### Pull 模式

1. 本地运行：`bun tools/hub-start.ts --course <course>`
2. 从终端输出的 `📋 Kaggle 粘贴用` 部分复制 HUB_URL 和 TOKEN
3. 在本 notebook 的 `⚙️ 参数配置` 中填入
4. 设置 `MODE = "pull"`
5. 依次运行各单元格

### Push 模式

1. 在本 notebook 设置 `MODE = "push"`，填入 PUSH_TOKEN
2. 运行 notebook，步骤 4 启动后等待 cloudflared 打印隧道 URL
3. 在 hub 侧 `rl-config.json` 的 nodes 中添加：
   ```json
   {
     "id": "gpu-worker",
     "url": "<cloudflared-tunnel-url>",
     "authKey": "<PUSH_TOKEN>",
     "concurrency": 1,
     "enabled": true,
     "gpu_push": true
   }
   ```
4. 本地运行 hub-start：`bun tools/hub-start.ts --course <course>`
   hub 会自动将 job 推送到 GPU 节点

### 可用课程

查看 `nn-training/curricula/` 目录获取完整列表。常见课程：

- `p1-onset` - 单敌近战
- `p4-onset` - 四面围攻（4 敌混编）
- `p4-horizon` - 水平进攻
- `s1` ~ `s5` - 专项技能课程
- `s-dodge` - 闪避训练
- `s3-balanced` - 平衡型课程